## Extract NOAA Data

This notebook downloads the shared Google Drive folder, load every `.psv` file, concatenates the NOAA rows into one dataframe, and saves the combined output as a parquet.

In [9]:
from pathlib import Path

import gdown
import pandas as pd

DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1qZHc7AxcL_1GaD2ZTN2n8O7lP7VmoLoh?usp=sharing"

NOTEBOOK_DIR = Path.cwd()
DATA_DIR = NOTEBOOK_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
COMBINED_DATA_DIR = DATA_DIR / "combined"

COMBINED_PARQUET_PATH = COMBINED_DATA_DIR / "sfo_noaa_combined.parquet"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
COMBINED_DATA_DIR.mkdir(parents=True, exist_ok=True)

### Download the Google Drive folder

In [4]:
downloaded_paths = gdown.download_folder(
    url=DRIVE_FOLDER_URL,
    output=str(RAW_DATA_DIR),
    quiet=False,
    use_cookies=False,
)

downloaded_paths = downloaded_paths

Retrieving folder contents


Processing file 1uiyqvqDJbWGmkm8_FNe621q6vIO6YAYU GHCNh_USW00023234_2022.psv
Processing file 1WP-Uqv2OcWlyHywI5vI9O2mrVNwg3ehl GHCNh_USW00023234_2023.psv
Processing file 1l_8LFjJFF8A2DKfE27pELincYqPe_CR7 GHCNh_USW00023234_2024.psv
Processing file 16_HK6-lyJPcaVCgk8fD_P4r8boICvPGn GHCNh_USW00023234_2025.psv
Processing file 1VttV-JczJMeqAIL5TeFVsZkuyeWqvxgF GHCNh_USW00023234_2026.psv


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1uiyqvqDJbWGmkm8_FNe621q6vIO6YAYU
To: c:\Users\smeri\OneDrive\Desktop\MIDS Coursework\207-Summer26-FinalProject-MLModel\Data_Extraction\santi\data\raw\GHCNh_USW00023234_2022.psv
100%|██████████| 10.7M/10.7M [00:00<00:00, 33.8MB/s]
Downloading...
From: https://drive.google.com/uc?id=1WP-Uqv2OcWlyHywI5vI9O2mrVNwg3ehl
To: c:\Users\smeri\OneDrive\Desktop\MIDS Coursework\207-Summer26-FinalProject-MLModel\Data_Extraction\santi\data\raw\GHCNh_USW00023234_2023.psv
100%|██████████| 11.2M/11.2M [00:00<00:00, 25.8MB/s]
Downloading...
From: https://drive.google.com/uc?id=1l_8LFjJFF8A2DKfE27pELincYqPe_CR7
To: c:\Users\smeri\OneDrive\Desktop\MIDS Coursework\207-Summer26-FinalProject-MLModel\Data_Extraction\santi\data\raw\GHCNh_USW00023234_2024.psv
100%|██████████| 12.0M/12.0M [00:00<00:00, 39.2MB/s]
Downloading...
From: https://drive.google.com/

### Load and concatenate NOAA data

In [21]:
def read_noaa_psv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, sep="|", low_memory=False)

def normalize_parquet_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    text_columns = df.columns[df.dtypes.eq("object")]
    df[text_columns] = df[text_columns].astype("string")
    return df

psv_files = RAW_DATA_DIR.rglob("*.psv")

weather_df = pd.concat((read_noaa_psv(path) for path in psv_files), ignore_index=True)

if "DATE" in weather_df.columns:
    weather_df["DATE"] = pd.to_datetime(weather_df["DATE"], errors="coerce")
    weather_df = weather_df.sort_values("DATE", na_position="last").reset_index(drop=True)

weather_df = normalize_parquet_dtypes(weather_df)

In [22]:
display(weather_df)

,STATION,Station_name,DATE,Year,Month,Day,Hour,Minute,LATITUDE,LONGITUDE,...,precipitation_24_hour_Quality_Code,precipitation_24_hour_Report_Type,precipitation_24_hour_Source_Code,precipitation_24_hour_Source_Station_ID,REM,REM_Measurement_Code,REM_Quality_Code,REM_Report_Type,REM_Source_Code,REM_Source_Station_ID
0,USW00023234,SAN FRANCISCO INTL AP,2022-01-01 00:00:00,2022,1,1,0,0,37.6197,-122.3656,...,NaN,NaN,NaN,NaN,SYN08072494 32566 42915 10117 20044 30094 4012...,NaN,NaN,FM12,223.0,ICAO-KSFO
1,USW00023234,SAN FRANCISCO INTL AP,2022-01-01 00:56:00,2022,1,1,0,56,37.6197,-122.3656,...,NaN,NaN,NaN,NaN,MET09812/31/21 16:56:03 METAR KSFO 010056Z 300...,NaN,NaN,FM15,343.0,724940-23234
2,USW00023234,SAN FRANCISCO INTL AP,2022-01-01 01:56:00,2022,1,1,1,56,37.6197,-122.3656,...,NaN,NaN,NaN,NaN,MET09812/31/21 17:56:03 METAR KSFO 010156Z 300...,NaN,NaN,FM15,343.0,724940-23234
3,USW00023234,SAN FRANCISCO INTL AP,2022-01-01 02:56:00,2022,1,1,2,56,37.6197,-122.3656,...,NaN,NaN,NaN,NaN,MET10112/31/21 18:56:03 METAR KSFO 010256Z 290...,NaN,NaN,FM15,343.0,724940-23234
4,USW00023234,SAN FRANCISCO INTL AP,2022-01-01 03:56:00,2022,1,1,3,56,37.6197,-122.3656,...,NaN,NaN,NaN,NaN,MET09812/31/21 19:56:03 METAR KSFO 010356Z 310...,NaN,NaN,FM15,343.0,724940-23234
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89171,USW00023234,SAN FRANCISCO INTL AP,2026-06-09 14:21:00,2026,6,9,14,21,37.6197,-122.3656,...,NaN,NaN,NaN,NaN,KSFO 091421Z 34006KT 10SM FEW006 BKN013 OVC043...,NaN,NaN,FM16,413.0,ICAO-KSFO
89172,USW00023234,SAN FRANCISCO INTL AP,2026-06-09 14:56:00,2026,6,9,14,56,37.6197,-122.3656,...,NaN,NaN,NaN,NaN,KSFO 091456Z 33007KT 10SM FEW006 BKN013 OVC045...,NaN,NaN,FM15,413.0,ICAO-KSFO
89173,USW00023234,SAN FRANCISCO INTL AP,2026-06-09 15:43:00,2026,6,9,15,43,37.6197,-122.3656,...,NaN,NaN,NaN,NaN,KSFO 091543Z 33007KT 10SM FEW006 SCT014 BKN039...,NaN,NaN,FM16,413.0,ICAO-KSFO
89174,USW00023234,SAN FRANCISCO INTL AP,2026-06-09 15:56:00,2026,6,9,15,56,37.6197,-122.3656,...,NaN,NaN,NaN,NaN,KSFO 091556Z 33006KT 10SM FEW006 SCT014 BKN039...,NaN,NaN,FM15,413.0,ICAO-KSFO


## Save the combined dataset

In [23]:
weather_df.to_parquet(COMBINED_PARQUET_PATH, index=False)
output_path = COMBINED_PARQUET_PATH

print(f"Wrote combined NOAA data to: {output_path}")

Wrote combined NOAA data to: c:\Users\smeri\OneDrive\Desktop\MIDS Coursework\207-Summer26-FinalProject-MLModel\Data_Extraction\santi\data\combined\sfo_noaa_combined.parquet
